# Avance 4 · Regresión de la potencia con huellas

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04b](04b_Regresion_actividad.ipynb). Repite sus modelos de regresión con las **mismas filas, los mismos pliegues y las mismas semillas**, y cambia sobre todo una cosa: qué información de la estructura reciben. El conjunto de modelos cambia en una sola pieza: el `SVR` con kernel no entra a los brazos con huellas, porque un solo ajuste con esas columnas pasó de 27 minutos, y en su lugar hay un `LinearSVR` para que la familia de márgenes siga representada. Son siete modelos, seis en cada brazo.

Los modelos del 03, del 04 y del 04b se entrenaron solo con descriptores de RDKit, que resumen la molécula entera (su tamaño, su polaridad, cuántos anillos tiene). La decisión [1.3](../docs/decisiones.md#d1-3) contemplaba también las huellas **Sort & Slice**, que dicen qué fragmentos concretos tiene la molécula y son la base para explicar una predicción a nivel de subestructura. Aquí medimos qué cambia al agregarlas. El 04b sigue siendo el cuaderno que se entrega; este lo acompaña.

## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import concordancia_harrell, rmse_exactas
from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.features.seleccion import DepuracionDescriptores, SeleccionL1
from src.models.pisos import MediaPorDiana

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Las huellas y el código que las arma

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura (el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4) y un 1 si la molécula la tiene. De todas las subestructuras que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto.

**El vocabulario se aprende en cada pliegue, con sus filas de entrenamiento.** Es lo que más importa de este cuaderno. Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten: si se cuentan con todas, la validación ya influyó en qué columnas existen, y eso es una fuga que no aparece en ninguna métrica y hace que las huellas parezcan mejores de lo que son. Por eso las moléculas se leen una sola vez (una columna por cada subestructura que aparece en alguna) y la elección de las 2,048 ocurre dentro de la canalización, en cada ajuste.

Dos diferencias con el `SortAndSlice` de `src/features/molecular.py`, que cuenta por molécula sobre una lista fija: aquí la frecuencia se cuenta por fila (en la fase 2 un compuesto aparece en 1.18 pares en promedio) y la elección se repite en cada ajuste. `tests/test_huellas.py` comprueba que, cuando cada molécula aparece una vez, el vocabulario y la matriz son idénticos a los de `SortAndSlice`.

**Lo que se ejecuta es el código de `src/`**, que importa la celda de arriba. Se muestra aquí para que se lea sin abrir otro archivo, y la prueba `tests/test_04c_codigo.py` comprueba que lo mostrado y lo importado sean idénticos.

```python
def identificadores_morgan(smiles, radio: int = 2) -> tuple[sparse.csr_matrix, np.ndarray]:
    """Matriz binaria dispersa (filas × subestructuras distintas) y los identificadores.

    Una columna por subestructura, sin plegar, ordenadas por identificador. Cada SMILES se lee
    una vez aunque aparezca en varias filas. Falla si alguno no se puede leer: quitarlo en
    silencio desalinearía la matriz con la de descriptores.
    """
    smiles = np.asarray(smiles, dtype=object)
    unicos, inverso = np.unique(smiles, return_inverse=True)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radio)
    ids_por_molecula = []
    ilegibles = []
    for s in unicos:
        mol = a_mol(s)
        if mol is None:
            ilegibles.append(s)
            ids_por_molecula.append(np.empty(0, dtype=np.int64))
        else:
            ids_por_molecula.append(
                np.fromiter(gen.GetSparseCountFingerprint(mol).GetNonzeroElements().keys(),
                            dtype=np.int64))
    if ilegibles:
        raise ValueError(f"{len(ilegibles)} SMILES no se pueden leer (el primero: {ilegibles[0]!r})")
    identificadores = np.unique(np.concatenate(ids_por_molecula))
    columnas = np.concatenate([np.searchsorted(identificadores, ids) for ids in ids_por_molecula])
    filas = np.repeat(np.arange(len(unicos)), [len(ids) for ids in ids_por_molecula])
    por_molecula = sparse.csr_matrix(
        (np.ones(len(columnas), dtype=np.uint8), (filas, columnas)),
        shape=(len(unicos), len(identificadores)))
    return por_molecula[inverso], identificadores

def agrupar_iguales(huellas: sparse.csr_matrix, identificadores: np.ndarray
                    ) -> tuple[sparse.csr_matrix, np.ndarray, list[tuple[int, ...]]]:
    """Deja una sola columna por cada grupo de subestructuras que valen lo mismo en todas las filas.

    Dos subestructuras que siempre aparecen juntas (el `S` y el `O=S` de un sulfonilo, por ejemplo)
    son la misma variable para cualquier modelo, y si ambas entran el modelo reparte su peso entre
    ellas de forma arbitraria. No se mira la etiqueta, y lo que es idéntico en todas las filas lo es
    también en cualquier subconjunto, así que hacerlo una vez sobre todo el conjunto no filtra
    información de la validación. Se queda la primera columna de cada grupo y se devuelve, para cada
    columna conservada, la tupla con los identificadores de todas las que representa (la propia
    primero), para poder nombrar la variable con todas sus subestructuras.
    """
    columnas = huellas.tocsc()
    columnas.sort_indices()
    grupos: dict[bytes, list[int]] = {}
    for j in range(columnas.shape[1]):
        grupos.setdefault(columnas.indices[columnas.indptr[j]:columnas.indptr[j + 1]].tobytes(),
                          []).append(j)
    conservadas = [g[0] for g in grupos.values()]
    equivalentes = [tuple(int(i) for i in identificadores[g]) for g in grupos.values()]
    return columnas[:, conservadas].tocsr(), identificadores[conservadas], equivalentes

class AgregarHuellas(BaseEstimator, TransformerMixin):
    """Entrada mixta → matriz densa con las columnas `densas` y las `n_bits` huellas más frecuentes.

    `X` es una matriz dispersa cuyas primeras columnas son las densas (descriptores y diana) y el
    resto es la salida de `identificadores_morgan`. En `fit` cuenta cuántas filas de entrenamiento
    tiene cada subestructura y se queda con las `n_bits` más frecuentes (los empates se resuelven por
    la posición de la columna, que sigue el orden del identificador, para que sea reproducible).
    `transform` devuelve las columnas densas indicadas y esas huellas, en `float32`.

    Con `densas=()` solo salen las huellas.
    """

    def __init__(self, n_densas: int, densas: tuple[int, ...] | None = None, n_bits: int = 2048):
        self.n_densas = n_densas
        self.densas = densas
        self.n_bits = n_bits

    def fit(self, X, y=None):
        huellas = sparse.csr_matrix(X)[:, self.n_densas:]
        frecuencia = np.asarray(huellas.sum(axis=0)).ravel()
        orden = np.lexsort((np.arange(len(frecuencia)), -frecuencia.astype(np.int64)))
        usadas = orden[:self.n_bits]
        self.columnas_ = usadas[frecuencia[usadas] > 0]
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X)
        densas = np.arange(self.n_densas) if self.densas is None else np.asarray(self.densas, int)
        partes = [X[:, densas].toarray().astype(np.float32),
                  X[:, self.n_densas:][:, self.columnas_].toarray().astype(np.float32)]
        return np.hstack(partes)
```

**El piso por diana.** El 04b solo tiene un piso, predecir siempre el promedio de todas las mediciones. Pero el corte de 10 µM no significa lo mismo en cada diana (deja a `LRRK2` en 98.1% de activos y a `NFE2L2` en 50.7%), y un modelo que solo sepa contra qué diana se midió ya supera ese promedio sin entender química. `MediaPorDiana` predice la potencia media de la diana de cada par con las filas de entrenamiento del pliegue, sin mirar los descriptores, y revienta si las columnas que se le pasan no son un one-hot, porque con las posiciones equivocadas daría la media global en todas las filas, una cifra creíble y falsa; lo que un modelo gane sobre ese piso es lo que le aporta la estructura de la molécula.

```python
class MediaPorDiana(BaseEstimator, RegressorMixin):
    """Predice la media de la variable objetivo de la diana de cada fila.

    `columnas` son las posiciones de las columnas de diana (una por diana, con 1 en la que
    corresponde y 0 en las demás). Una diana que no aparece en las filas de ajuste recibe la
    media de todas ellas. Si esas columnas no son un one-hot (cada fila con un solo 1 y el
    resto 0) revienta, porque con las posiciones equivocadas el piso devolvería la media global
    en todas las filas: una cifra creíble y falsa.
    """

    def __init__(self, columnas: tuple[int, ...]):
        self.columnas = columnas

    def _dianas(self, X):
        diana = np.asarray(X)[:, list(self.columnas)]
        if not (np.isin(diana, (0, 1)).all() and (diana.sum(axis=1) == 1).all()):
            raise ValueError(
                "las columnas de diana no son un one-hot: cada fila debe tener un solo 1 y el "
                "resto 0; revisa las posiciones que se le pasaron"
            )
        return diana

    def fit(self, X, y):
        diana = self._dianas(X)
        y = np.asarray(y, dtype=float)
        self.media_global_ = float(y.mean())
        n = diana.sum(axis=0)
        suma = diana.T @ y
        self.medias_ = np.where(n > 0, suma / np.maximum(n, 1), self.media_global_)
        return self

    def predict(self, X):
        return self._dianas(X) @ self.medias_
```

La depuración de descriptores (brazo `depurados`, más abajo) usa dos filtros que ya existían, `casi_constantes` y `filtro_correlacion`, y la selección por coeficientes (brazo `seleccionados`) usa un Lasso. Las dos se ajustan también dentro de cada pliegue:

```python
def poder_spearman(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """|Spearman| de cada columna contra un objetivo continuo: la versión de `poder_univariado`
    para una regresión. Una columna constante da 0."""
    rx = np.apply_along_axis(rankdata, 0, X)
    ry = rankdata(y)
    rx = rx - rx.mean(axis=0)
    ry = ry - ry.mean()
    den = np.sqrt((rx ** 2).sum(axis=0) * (ry ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (rx * ry[:, None]).sum(axis=0) / den
    return np.abs(np.nan_to_num(r))

class DepuracionDescriptores(BaseEstimator, TransformerMixin):
    """Los dos filtros de siempre, pero aprendidos con las filas de entrenamiento de cada pliegue.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    Quita las casi constantes (un valor reúne `frecuencia_modal` de las filas) y, de cada par con
    |Spearman| mayor que `umbral`, la que menos se relaciona con el objetivo. Las `protegidas`
    (posiciones dentro de los descriptores) ganan cualquier par. Es el criterio de
    `casi_constantes` y `filtro_correlacion`; lo que cambia es dónde se ajusta.
    """

    def __init__(self, n_descriptores: int, protegidas: tuple[int, ...] = (),
                 frecuencia_modal: float = 0.99, umbral: float = 0.95):
        self.n_descriptores = n_descriptores
        self.protegidas = protegidas
        self.frecuencia_modal = frecuencia_modal
        self.umbral = umbral

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        variables = np.flatnonzero(casi_constantes(desc, self.frecuencia_modal))
        self.quitadas_constantes_ = np.setdiff1d(np.arange(self.n_descriptores), variables)
        sub = desc[:, variables]
        en_sub = tuple(int(np.flatnonzero(variables == p)[0]) for p in self.protegidas
                       if p in variables)
        sin_redundancia = filtro_correlacion(sub, poder_spearman(sub, np.asarray(y, float)),
                                             self.umbral, en_sub)
        self.descriptores_ = variables[sin_redundancia]
        self.quitadas_redundantes_ = variables[~sin_redundancia]
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]

class SeleccionL1(BaseEstimator, TransformerMixin):
    """Se queda con los descriptores a los que una penalización L1 no les lleva el peso a cero.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    En cada ajuste escala los descriptores, entrena un Lasso contra el objetivo con las filas de
    entrenamiento y conserva los de coeficiente distinto de cero. Las `protegidas` se quedan
    siempre. `alpha` es un hiperparámetro que se fija de antemano; no se elige mirando la
    validación.
    """

    def __init__(self, n_descriptores: int, alpha: float = 0.01, protegidas: tuple[int, ...] = ()):
        self.n_descriptores = n_descriptores
        self.alpha = alpha
        self.protegidas = protegidas

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        lasso = Lasso(alpha=self.alpha, max_iter=5000).fit(
            StandardScaler().fit_transform(desc), np.asarray(y, float))
        elegidas = set(np.flatnonzero(lasso.coef_ != 0)) | set(self.protegidas)
        self.descriptores_ = np.array(sorted(elegidas), dtype=int)
        self.quitadas_ = np.setdiff1d(np.arange(self.n_descriptores), self.descriptores_)
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]
```

### Los datos

Son los mismos pares, con el mismo criterio, que el 04b: entrenamiento y validación (la prueba reservada no se toca), sin los pares que no informan nada, y con la diana como columnas de 0 y 1. Los modelos se entrenan con las mediciones exactas y la validación incluye las censuradas.

In [2]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")

56,411 pares de entrenamiento y validación · 170 sin información y salen
56,241 pares · 46,100 con potencia exacta (82.0%) · 10,141 solo con cotas


### Las huellas de los pares

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada pliegue. Antes se agrupan las subestructuras que valen lo mismo en todos los pares (porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo): se queda una sola columna y se guarda, en `equivalentes`, el nombre de todas las que representa. Eso no mira la potencia, y lo que es idéntico en todas las filas lo es en cualquier pliegue, así que no hay fuga. Así las 2,048 columnas no se gastan en la misma variable repetida y una atribución no reparte su peso entre copias. Las columnas de descriptores y de diana van primero y las huellas después, en una sola matriz dispersa.

In [3]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(d["smiles"])
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
N_DESC = len(cols)                       # descriptores
N_DENSAS = X.shape[1]                    # descriptores + diana
DIANA = tuple(range(N_DESC, N_DENSAS))
X_mixta = sparse.hstack([sparse.csr_matrix(X), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} pares; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")

51,616 subestructuras distintas en 56,241 pares; 18,531 valían lo mismo que otra y quedan 33,085 columnas (8668 de ellas representan a más de una) · 4.0 s


## 2. Cinco brazos con los modelos del 04b

| Brazo | Qué recibe el modelo | Para qué está |
|:--|:--|:--|
| `descriptores` | 201 descriptores y la diana | Es el 04b. Es el **control**: tiene que repetir sus cifras |
| `depurados` | Los descriptores sin las columnas casi constantes ni las redundantes (|Spearman| mayor que 0.95), y la diana | Responde si 201 descriptores son demasiados. TPSA y los donadores de hidrógeno se protegen, porque son las dos propiedades con las que se comprueba que una explicación tenga sentido |
| `seleccionados` | Los descriptores a los que un Lasso (penalización L1, `alpha = 0.01`) no les lleva el peso a cero, y la diana | La otra forma de preguntar si sobran columnas: que sea un modelo el que elija, con sus coeficientes. Se ajusta dentro de cada pliegue, y no con los resultados del Avance 4, porque elegir variables mirando una validación y volver a evaluar en ella da cifras mejores de lo que son. `alpha` se fijó de antemano (conserva unos 90 de los 201 en un pliegue de entrenamiento, sin mirar ninguna validación) |
| `descriptores+huellas` | Los 201 descriptores, las 2,048 huellas y la diana | Lo que se quiere medir: qué agregan los fragmentos a lo que ya había |
| `huellas` | Solo las 2,048 huellas y la diana | Cuánto dice la estructura por sí sola, sin los descriptores |

Los modelos y sus hiperparámetros son los del 04b; la explicación de lo que hace cada uno está allí. El `LinearSVR` es el único que el 04b no tiene: entra para que la familia de márgenes esté representada en los brazos con huellas, de los que el `SVR` con kernel queda fuera por costo, y se corre también en los tres brazos sin huellas para tener con qué compararlo. Así cada brazo lleva seis modelos: los tres sin huellas con `SVR` y sin él los dos que las llevan. Los que dependen de distancias o de magnitudes llevan el escalado dentro de la canalización, así que cada pliegue aprende el suyo. En las columnas de huellas, que ya valen 0 o 1, el escalado mínimo-máximo no cambia nada.

Además de los cinco brazos, se miden dos pisos que no ven la química, **una sola vez, con los mismos pliegues**: `media` (el promedio de todo, el piso del 04b) y `media_por_diana` (el promedio de cada diana). Las dos tablas de resultados los muestran aparte, y una tabla más resta el segundo a cada modelo para ver cuánto de la concordancia queda una vez que se sabe la diana.

**Lo que se fija antes de correr.** La regla de mejora no está pre-registrada en sentido estricto: se fijó después de explorar un pliegue del 04b, y el umbral sale de la dispersión que ya se había visto allí. Lo que sí se cumple es que quedó escrita antes de ejecutar este cuaderno y no se ajusta después.

- **La métrica principal es la concordancia de Harrell**, igual que en el 04b, y se compara **pliegue por pliegue contra el brazo `descriptores`**: las diferencias se calculan sobre las mismas 15 evaluaciones (3 semillas × 5 pliegues), no entre promedios.
- **Un brazo mejora a un modelo solo si** la diferencia media de concordancia supera **0.007** (la mayor dispersión entre evaluaciones que se vio en el 04b) **y** es positiva en al menos **12 de las 15** evaluaciones. Si no, se reporta como «sin mejora medible». La comparación decisiva es una por modelo: el agregado de todas las moléculas contra `descriptores`. Los estratos por procedencia se muestran, pero son exploratorios y no deciden. Si las huellas no mejoran la métrica, el argumento a favor de usarlas sería la explicabilidad, y así se diría; no se presentaría como una mejora.
- **Contra el sobreajuste**, además de los pliegues agrupados por esqueleto (un modelo que memoriza fragmentos de un esqueleto no gana nada en el pliegue que lo deja fuera), se mide la **brecha entre entrenamiento y validación** de cada modelo, y se incluyen los brazos `depurados` y `seleccionados`.

Tres avisos sobre la columna de tiempos. Cronometra **solo el ajuste**, no la predicción, así que el `kNN` sale en casi cero: no aprende nada al ajustar y todo su costo está en buscar vecinos al predecir, que esta columna no ve. Los brazos con huellas no resultan mucho más caros que `descriptores` precisamente porque el `SVR`, el más lento con los descriptores, no entra en ellos; el modelo que domina allí es el `MLP`. Y cada ajuste corre con un solo hilo, para que la columna se pueda comparar entre modelos: no es el tiempo que tardaría aprovechando la máquina.

In [4]:
MODELOS = {
    "ridge":          make_pipeline(MinMaxScaler(), Ridge()),
    "arbol":          DecisionTreeRegressor(min_samples_leaf=5),
    "knn":            make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "bayesian_ridge": make_pipeline(StandardScaler(), BayesianRidge()),
    "svr":            make_pipeline(MinMaxScaler(), SVR()),
    # Misma familia de margenes con nucleo lineal, para que la reja no quede incompleta
    # en los brazos anchos. max_iter alto porque liblinear no converge en 5,000.
    "linear_svr":     make_pipeline(MinMaxScaler(), LinearSVR(dual="auto", max_iter=20000)),
    "mlp":            make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
REFERENCIA = {"media": DummyRegressor(),                 # el piso del 04b: el promedio de todo
              "media_por_diana": MediaPorDiana(DIANA)}  # el piso que sabe la diana y nada de química
# El SVR con nucleo no cabe con 2,275 columnas: un solo ajuste paso de 27 minutos, mas de 15 h
# solo estos dos brazos. La omision cae del lado conservador, porque le quita a las huellas un
# posible ganador en el brazo que se quiere defender. `linear_svr` cubre la familia en los cinco.
ANCHOS = ("descriptores+huellas", "huellas")
SEMILLAS = (0, 1, 2)
CORTE = 5.0                                      # pActividad 5 = 10 µM, el mismo corte del 04
N_BITS = 2048                                    # tamaño del vocabulario de Sort & Slice
ALFA_L1 = 0.01                                   # penalización del Lasso; fija, no se elige con la validación
PROTEGIDAS = tuple(cols.index(c) for c in ("TPSA", "NumHDonors"))   # posiciones dentro de los descriptores
# Regla de mejora, cerrada antes de ejecutar: 0.007 es la mayor dispersion entre evaluaciones del
# 04b, y se fijo despues de explorar un pliegue, asi que no esta pre-registrada (ver la celda de registro).
UMBRAL_MEJORA = 0.007                            # diferencia de concordancia que hay que superar
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
MUESTRA_ENTRENAMIENTO = 5000                     # filas de entrenamiento para medir la brecha

# Cada brazo: la matriz que recibe y cómo se arma su canalización a partir del modelo del 04b.
BRAZOS = {
    "descriptores":         (X,       lambda m: m),
    "depurados":            (X,       lambda m: make_pipeline(DepuracionDescriptores(N_DESC, PROTEGIDAS), m)),
    "seleccionados":        (X,       lambda m: make_pipeline(SeleccionL1(N_DESC, ALFA_L1, PROTEGIDAS), m)),
    "descriptores+huellas": (X_mixta, lambda m: make_pipeline(AgregarHuellas(N_DENSAS, n_bits=N_BITS), m)),
    "huellas":              (X_mixta, lambda m: make_pipeline(AgregarHuellas(N_DENSAS, DIANA, N_BITS), m)),
}

In [5]:
def medir(inf, sup, clase, pred):
    """Concordancia con todas las mediciones; Spearman y RMSE con las exactas; MCC y AUROC de la clase derivada."""
    exa = inf == sup
    pe = pred[exa]
    rho = np.nan if np.ptp(pe) == 0 else spearmanr(inf[exa], pe).statistic   # nan si predice una constante
    rmse, _ = rmse_exactas(inf, sup, pred)
    fila = {"concordancia": concordancia_harrell(inf, sup, pred), "spearman": rho, "rmse": rmse,
            "mcc": np.nan, "auroc": np.nan}
    decide = ~np.isnan(clase)
    if len(np.unique(clase[decide])) == 2:
        fila["mcc"] = matthews_corrcoef(clase[decide], pred[decide] >= CORTE)
        fila["auroc"] = roc_auc_score(clase[decide], pred[decide])
    return fila


filas = []
elegidas = {}                                    # qué descriptores dejó cada brazo de selección en cada ajuste
quitadas = {}                                    # y cuáles quitó cada paso, para contar cómo se fueron quitando
estratificar = np.where(np.isnan(y_cls), -1, y_cls).astype(int)
inicio = time.perf_counter()
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)         # los mismos pliegues del 04b
    for k, (ajuste, val) in enumerate(cv.split(X, estratificar, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]                                    # se entrena solo con las exactas
        # Un subconjunto del entrenamiento para medir la brecha; el mismo en todos los brazos.
        muestra = np.random.default_rng(semilla * 10 + k).choice(
            ajuste_ex, min(MUESTRA_ENTRENAMIENTO, len(ajuste_ex)), replace=False)
        for brazo, (matriz, armar) in BRAZOS.items():
            X_aj, X_val, X_mue = matriz[ajuste_ex], matriz[val], matriz[muestra]
            # Los pisos no ven la química, así que se miden una vez, en el brazo de control.
            modelos_brazo = {n: m for n, m in MODELOS.items()
                             if not (brazo in ANCHOS and n == "svr")}
            for nombre, modelo in {**modelos_brazo,
                                   **(REFERENCIA if brazo == "descriptores" else {})}.items():
                m = armar(clone(modelo))
                # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
                m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
                # `catch_warnings` silencia lo que salga del ajuste, asi que la convergencia se
                # guarda en una columna en vez de perderse en el log de una corrida de horas.
                with warnings.catch_warnings(record=True) as avisos:
                    warnings.simplefilter("always", ConvergenceWarning)
                    reloj = time.perf_counter()
                    m.fit(X_aj, inferior[ajuste_ex])
                    seg = time.perf_counter() - reloj
                convergio = not any(issubclass(a.category, ConvergenceWarning) for a in avisos)
                pred = m.predict(X_val)
                pred_mue = m.predict(X_mue)
                paso = m.steps[0][1] if hasattr(m, "steps") else m
                if hasattr(paso, "descriptores_") and nombre == "ridge":      # igual para todos: no depende del modelo
                    elegidas[(brazo, semilla, k)] = [cols[i] for i in paso.descriptores_]
                    quitadas[(brazo, semilla, k)] = {
                        paso_: [cols[i] for i in getattr(paso, atributo, [])]
                        for paso_, atributo in (("casi constantes", "quitadas_constantes_"),
                                                ("redundantes", "quitadas_redundantes_"),
                                                ("Lasso en cero", "quitadas_"))}
                base = {"brazo": brazo, "modelo": nombre, "semilla": semilla, "pliegue": k,
                        "seg": seg, "convergio": convergio}
                ent = {"ent_concordancia": concordancia_harrell(inferior[muestra], superior[muestra], pred_mue),
                       "ent_rmse": rmse_exactas(inferior[muestra], superior[muestra], pred_mue)[0]}
                filas.append({**base, "estrato": "todos", **ent,
                              **medir(inferior[val], superior[val], y_cls[val], pred)})
                for e in np.unique(procedencia[val]):
                    sel = procedencia[val] == e
                    ys = y_cls[val][sel]
                    ys = ys[~np.isnan(ys)]
                    if min(ys.sum(), (1 - ys).sum()) >= 20:         # estrato con 20 casos de cada clase
                        filas.append({**base, "estrato": e,
                                      **medir(inferior[val][sel], superior[val][sel], y_cls[val][sel], pred[sel])})
            print(f"semilla {semilla} · pliegue {k} · {brazo:<22} {(time.perf_counter() - inicio) / 60:6.1f} min", flush=True)
res = pd.DataFrame(filas)
print(f"{len(res):,} filas: {res['brazo'].nunique()} brazos × {res['modelo'].nunique()} modelos × "
      f"{len(SEMILLAS)} semillas × 5 pliegues, por estrato")

semilla 0 · pliegue 1 · descriptores              1.9 min


semilla 0 · pliegue 1 · depurados                 3.5 min


semilla 0 · pliegue 1 · seleccionados             4.8 min


semilla 0 · pliegue 1 · descriptores+huellas      6.1 min


semilla 0 · pliegue 1 · huellas                   7.1 min


...

semilla 2 · pliegue 5 · depurados               115.2 min


semilla 2 · pliegue 5 · seleccionados           117.0 min


semilla 2 · pliegue 5 · descriptores+huellas    118.6 min


semilla 2 · pliegue 5 · huellas                 120.1 min


2,100 filas: 5 brazos × 9 modelos × 3 semillas × 5 pliegues, por estrato


## 3. Resultados

### Cómo se fueron quitando variables

Los modelos del 04 y del 04b se entrenaron con **todos** los descriptores que dejó la limpieza, sin quitar ninguno más. Esta tabla cuenta cada paso por el que pasa una columna, desde las que calcula RDKit hasta las que ve un modelo, y quién la quitó:

- **Limpieza.** Se hace una sola vez, sin mirar la potencia: quita las columnas constantes, las que tienen valores faltantes o infinitos y las que se salen del rango de `float32`. Es de donde salen los 201 descriptores que usan el 04b y el brazo `descriptores`.
- **`depurados`.** Se ajusta en cada pliegue, en dos pasos: primero las casi constantes (un solo valor reúne el 99% de las filas) y luego las redundantes (de cada par con |Spearman| mayor que 0.95 se queda la que más se relaciona con la potencia).
- **`seleccionados`.** Se ajusta en cada pliegue: se queda con lo que un Lasso no manda a cero.

En los dos casos, TPSA y NumHDonors no se pueden quitar. Como cada pliegue elige con sus propias filas, las cuentas son un promedio con el mínimo y el máximo entre las 15 evaluaciones, y después se ve cuáles descriptores sobreviven siempre.

In [6]:
from src.features.molecular import limpiar_descriptores

crudos = pd.read_parquet(PROCESADOS / "fase2_descriptores.parquet").drop(columns="smiles")
limpio, motivos = limpiar_descriptores(crudos, verbose=False)
assert list(limpio.columns) == cols, "las columnas de la matriz no son las que da la limpieza"


def cuantas(brazo, paso):
    v = [len(q[paso]) for (b, _, _), q in quitadas.items() if b == brazo]
    return f"{np.mean(v):.0f} ({min(v)}–{max(v)})", np.mean(v)


q_const, m_const = cuantas("depurados", "casi constantes")
q_red, m_red = cuantas("depurados", "redundantes")
q_l1, m_l1 = cuantas("seleccionados", "Lasso en cero")
por_motivo = ", ".join(f"{k.replace('_', ' ')} {len(v)}" for k, v in motivos.items() if v)
embudo = pd.DataFrame([
    ("RDKit calcula", "", "", len(crudos.columns)),
    ("Limpieza, una vez", por_motivo, f"{len(crudos.columns) - N_DESC}", N_DESC),
    ("`depurados`: casi constantes", "un valor reúne el 99% de las filas", q_const, f"{N_DESC - m_const:.0f}"),
    ("`depurados`: redundantes", "|Spearman| > 0.95 con otra", q_red, f"{N_DESC - m_const - m_red:.0f}"),
    ("`seleccionados`: Lasso", f"coeficiente cero con alpha = {ALFA_L1}", q_l1, f"{N_DESC - m_l1:.0f}"),
], columns=["paso", "criterio", "quita (media y rango)", "quedan"]).set_index("paso")
embudo

,criterio,quita (media y rango),quedan
paso,,,
RDKit calcula,,,217
"Limpieza, una vez","constantes 3, con nan 12, no finitas 12, fuera...",16,201
`depurados`: casi constantes,un valor reúne el 99% de las filas,23 (21–24),178
`depurados`: redundantes,|Spearman| > 0.95 con otra,24 (23–25),154
`seleccionados`: Lasso,coeficiente cero con alpha = 0.01,106 (99–110),95


Y los descriptores que sobreviven en todas las evaluaciones, que son los que más se pueden defender como parte de la explicación:

In [7]:
resumen = []
for brazo in ("depurados", "seleccionados"):
    listas = [v for (b, _, _), v in elegidas.items() if b == brazo]
    veces = pd.Series([c for v in listas for c in v]).value_counts()
    resumen.append({"brazo": brazo, "evaluaciones": len(listas),
                    "descriptores_que_deja": f"{min(map(len, listas))}–{max(map(len, listas))} de {N_DESC}",
                    "en_todas": int((veces == len(listas)).sum())})
display(pd.DataFrame(resumen).set_index("brazo"))
en_todas_l1 = [c for c, n in pd.Series([c for (b, _, _), v in elegidas.items() if b == "seleccionados" for c in v]).value_counts().items()
               if n == sum(1 for (b, _, _) in elegidas if b == "seleccionados")]
print(f"Descriptores que el Lasso conserva en todas las evaluaciones ({len(en_todas_l1)}):")
print(", ".join(en_todas_l1))

,evaluaciones,descriptores_que_deja,en_todas
brazo,,,
depurados,15,152–157 de 201,149
seleccionados,15,91–102 de 201,57


Descriptores que el Lasso conserva en todas las evaluaciones (57):
MaxAbsEStateIndex, qed, FpDensityMorgan3, AvgIpc, BalabanJ, PEOE_VSA10, PEOE_VSA3, PEOE_VSA5, PEOE_VSA7, PEOE_VSA8, PEOE_VSA9, SMR_VSA4, SlogP_VSA4, SlogP_VSA6, SlogP_VSA7, SlogP_VSA8, TPSA, EState_VSA3, EState_VSA5, EState_VSA8, VSA_EState9, NumAmideBonds, NumBridgeheadAtoms, NumHDonors, NumSaturatedCarbocycles, NumSpiroAtoms, NumUnspecifiedAtomStereoCenters, MolLogP, fr_ArN, fr_Ar_N, fr_C_S, fr_HOCCN, fr_Imine, fr_aldehyde, fr_alkyl_carbamate, fr_aniline, fr_aryl_methyl, fr_barbitur, fr_bicyclic, fr_epoxide, fr_ester, fr_halogen, fr_imidazole, fr_imide, fr_lactone, fr_nitrile, fr_oxazole, fr_oxime, fr_phenol, fr_piperdine, fr_priamide, fr_prisulfonamd, fr_pyridine, fr_sulfide, fr_sulfone, fr_thiazole, fr_urea


### El control: el brazo `descriptores` repite el 04b

Antes de leer nada, el brazo que no cambia nada tiene que dar lo que ya se registró. Si no coincide, algo en la preparación o en el entorno es distinto y los demás brazos no se pueden comparar con el 04b.

In [8]:
REGISTRO_04B = RAIZ / "experimentos" / "2026-10-25-artemio-2.9-comparativa-regresion-seis-modelos" / "metricas.json"
registrado = json.loads(REGISTRO_04B.read_text(encoding="utf-8"))["todos"]
# El pareo se deriva de lo que el 04b registro, no de `list(MODELOS)`: `linear_svr` es nuevo del
# 04c —entro para cubrir la familia de margenes en los brazos anchos, de donde se excluyo el SVR—
# y no tiene contraparte que comparar. Derivarlo evita que esto se rompa al agregar otro modelo.
pareados = [m for m in MODELOS if m in registrado]
nuevos = [m for m in MODELOS if m not in registrado]
aqui = res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")].groupby("modelo")["concordancia"].mean()
control = pd.DataFrame({"04b (registrado)": {m: registrado[m]["concordancia"] for m in pareados},
                        "04c (esta corrida)": aqui.loc[pareados]}).round(3)
control["diferencia"] = (control["04c (esta corrida)"] - control["04b (registrado)"]).round(3)
print(f"modelos nuevos en el 04c, sin contraparte en el 04b: {nuevos or 'ninguno'}")
control

modelos nuevos en el 04c, sin contraparte en el 04b: ['linear_svr']


,04b (registrado),04c (esta corrida),diferencia
ridge,0.691,0.691,0.0
arbol,0.670,0.670,0.0
knn,0.739,0.739,0.0
bayesian_ridge,0.692,0.692,0.0
svr,0.738,0.738,0.0
mlp,0.735,0.735,0.0


### Todos los brazos, por modelo

La concordancia media de cada modelo en cada brazo (la métrica principal), y en la segunda tabla cuánto tarda un ajuste.

In [9]:
def tabla(r, estrato="todos"):
    """Media por brazo y modelo. `desv_concordancia` es dispersión entre evaluaciones, no error estándar:
    las semillas comparten filas."""
    r = r[r["estrato"] == estrato]
    t = r.groupby(["brazo", "modelo"])[["concordancia", "spearman", "rmse", "mcc", "auroc", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby(["brazo", "modelo"])["concordancia"].std())
    return t.round(3)


tabla_todos = tabla(res)
orden_brazos = list(BRAZOS)
concordancia = tabla_todos["concordancia"].unstack("brazo")[orden_brazos]
# El orden lo da el control y no una lista a mano, para que no envejezca al agregar o quitar un
# modelo. Los pisos quedan fuera porque solo existen en el brazo de control.
orden_modelos = [m for m in concordancia["descriptores"].sort_values(ascending=False).index
                 if m in MODELOS]
concordancia.loc[orden_modelos]

brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
knn,0.739,0.739,0.731,0.756,0.755
svr,0.738,0.739,0.734,NaN,NaN
mlp,0.735,0.737,0.732,0.708,0.714
bayesian_ridge,0.692,0.692,0.688,0.728,0.726
ridge,0.691,0.691,0.688,0.724,0.722
linear_svr,0.690,0.690,0.687,0.722,0.720
arbol,0.670,0.672,0.670,0.684,0.716


In [10]:
# El SVR sale vacio en los dos brazos anchos: no se corrio ahi, a proposito.
tabla_todos["seg"].unstack("brazo")[orden_brazos].loc[orden_modelos].round(1)

brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
knn,0.0,0.5,0.3,0.2,0.1
svr,55.9,49.2,39.0,NaN,NaN
mlp,18.1,15.1,9.2,70.0,50.0
bayesian_ridge,0.1,0.5,0.3,2.9,2.5
ridge,0.0,0.5,0.2,0.5,0.2
linear_svr,0.9,1.0,0.4,7.6,2.2
arbol,2.1,2.1,1.3,5.5,3.8


### Los pisos, y cuánto queda una vez que se sabe la diana

La primera tabla son los dos pisos. La segunda resta la concordancia de `media_por_diana` a la de cada modelo y brazo, en el agregado y por procedencia: es la parte del resultado que no se explica por saber contra qué diana se midió.

In [11]:
pisos = tabla_todos.xs("descriptores", level="brazo").loc[list(REFERENCIA), ["concordancia", "spearman", "rmse", "mcc", "auroc"]]
display(pisos)

ganancia = {}
for e in ["todos"] + sorted(set(res["estrato"]) - {"todos"}):
    t_e = tabla(res, e)["concordancia"].unstack("brazo")[orden_brazos].loc[list(MODELOS)]
    piso_e = tabla(res, e).xs("descriptores", level="brazo").loc["media_por_diana", "concordancia"]
    ganancia[e] = (t_e - piso_e).round(3)
pd.concat(ganancia, names=["procedencia"])

,concordancia,spearman,rmse,mcc,auroc
modelo,,,,,
media,0.500,NaN,1.337,0.0,0.50
media_por_diana,0.642,0.384,1.238,0.0,0.72


brazo                       descriptores  depurados  seleccionados  \
procedencia modelo                                                   
todos       ridge                  0.049      0.049          0.046   
            arbol                  0.028      0.030          0.028   
            knn                    0.097      0.097          0.089   
            bayesian_ridge         0.050      0.050          0.046   
            svr                    0.096      0.097          0.092   
            linear_svr             0.048      0.048          0.045   
            mlp                    0.093      0.095          0.090   
mixto       ridge                 -0.002     -0.003          0.003   
            arbol                  0.015      0.025          0.027   
            knn                    0.048      0.054          0.030   
            bayesian_ridge        -0.007     -0.009          0.002   
            svr                    0.047      0.047          0.035   
            linear_svr             0.001      0.000          0.004   
            mlp                    0.053      0.059          0.069   
solo_chembl ridge                  0.043      0.043          0.040   
            arbol                  0.028      0.029          0.028   
            knn                    0.096      0.097          0.089   
            bayesian_ridge         0.044      0.043          0.040   
            svr                    0.093      0.094          0.089   
            linear_svr             0.042      0.041          0.039   
            mlp                    0.092      0.094          0.089   
solo_npass  ridge                  0.119      0.121          0.096   
            arbol                  0.095      0.101          0.093   
            knn                    0.061      0.062          0.049   
            bayesian_ridge         0.129      0.130          0.097   
            svr                    0.132      0.129          0.107   
            linear_svr             0.105      0.109          0.090   
            mlp                    0.148      0.146          0.117   

brazo                       descriptores+huellas  huellas  
procedencia modelo                                         
todos       ridge                          0.082    0.080  
            arbol                          0.042    0.074  
            knn                            0.114    0.113  
            bayesian_ridge                 0.086    0.084  
            svr                              NaN      NaN  
            linear_svr                     0.080    0.078  
            mlp                            0.066    0.072  
mixto       ridge                          0.031    0.033  
            arbol                          0.034    0.057  
            knn                            0.037    0.037  
            bayesian_ridge                 0.027    0.027  
            svr                              NaN      NaN  
            linear_svr                     0.034    0.039  
            mlp                            0.037    0.050  
solo_chembl ridge                          0.081    0.079  
            arbol                          0.042    0.074  
            knn                            0.112    0.111  
            bayesian_ridge                 0.084    0.082  
            svr                              NaN      NaN  
            linear_svr                     0.078    0.076  
            mlp                            0.069    0.074  
solo_npass  ridge                          0.087    0.063  
            arbol                          0.111    0.095  
            knn                            0.144    0.145  
            bayesian_ridge                 0.087    0.063  
            svr                              NaN      NaN  
            linear_svr                     0.094    0.074  
            mlp                            0.100    0.108

### ¿Mejoran las huellas? La comparación pliegue por pliegue

Para cada modelo y cada brazo se resta la concordancia del brazo `descriptores` en **la misma evaluación** (misma semilla, mismo pliegue). `mejor_en` cuenta en cuántas de las 15 evaluaciones el brazo gana, y `mejora` aplica la regla fijada arriba: diferencia media mayor que 0.007 **y** al menos 12 de 15.

In [12]:
def comparacion(r, estrato="todos"):
    """Diferencia de concordancia de cada brazo contra `descriptores`, evaluación por evaluación."""
    r = r[(r["estrato"] == estrato) & ~r["modelo"].isin(list(REFERENCIA))]
    p = r.pivot_table(index=["modelo", "semilla", "pliegue"], columns="brazo", values="concordancia")
    filas = []
    for brazo in orden_brazos[1:]:
        dif = (p[brazo] - p["descriptores"]).dropna()
        for modelo, v in dif.groupby("modelo"):
            filas.append({"brazo": brazo, "modelo": modelo, "evaluaciones": len(v),
                          "dif_media": v.mean(), "dif_desv": v.std(), "mejor_en": int((v > 0).sum())})
    t = pd.DataFrame(filas).set_index(["brazo", "modelo"])
    t["mejora"] = (t["dif_media"] > UMBRAL_MEJORA) & (t["mejor_en"] >= MINIMO_PLIEGUES)
    return t.round(3)


comparacion(res)

evaluaciones  dif_media  dif_desv  \
brazo                modelo                                              
depurados            arbol                     15      0.001     0.003   
                     bayesian_ridge            15     -0.000     0.001   
                     knn                       15      0.001     0.001   
                     linear_svr                15     -0.000     0.001   
                     mlp                       15      0.003     0.004   
                     ridge                     15     -0.000     0.001   
                     svr                       15      0.001     0.001   
seleccionados        arbol                     15     -0.000     0.011   
                     bayesian_ridge            15     -0.005     0.001   
                     knn                       15     -0.007     0.002   
                     linear_svr                15     -0.003     0.001   
                     mlp                       15     -0.003     0.006   
                     ridge                     15     -0.004     0.001   
                     svr                       15     -0.004     0.002   
descriptores+huellas arbol                     15      0.014     0.007   
                     bayesian_ridge            15      0.036     0.005   
                     knn                       15      0.017     0.004   
                     linear_svr                15      0.032     0.004   
                     mlp                       15     -0.027     0.009   
                     ridge                     15      0.033     0.005   
huellas              arbol                     15      0.046     0.008   
                     bayesian_ridge            15      0.033     0.005   
                     knn                       15      0.016     0.004   
                     linear_svr                15      0.030     0.005   
                     mlp                       15     -0.021     0.007   
                     ridge                     15      0.031     0.005   

                                     mejor_en  mejora  
brazo                modelo                            
depurados            arbol                  8   False  
                     bayesian_ridge         4   False  
                     knn                   13   False  
                     linear_svr             8   False  
                     mlp                   13   False  
                     ridge                  5   False  
                     svr                   13   False  
seleccionados        arbol                  7   False  
                     bayesian_ridge         0   False  
                     knn                    0   False  
                     linear_svr             0   False  
                     mlp                    5   False  
                     ridge                  0   False  
                     svr                    0   False  
descriptores+huellas arbol                 15    True  
                     bayesian_ridge        15    True  
                     knn                   15    True  
                     linear_svr            15    True  
                     mlp                    0   False  
                     ridge                 15    True  
huellas              arbol                 15    True  
                     bayesian_ridge        15    True  
                     knn                   15    True  
                     linear_svr            15    True  
                     mlp                    0   False  
                     ridge                 15    True

### Por procedencia

Un promedio puede esconder cómo le va a los productos naturales. En el 04b los seis modelos quedaron cerca del azar en los pares que solo tiene NPASS, y es ahí donde un fragmento común con compuestos de entrenamiento podría ayudar donde los descriptores no. También es donde más cuidado hay que tener: los fragmentos típicos de los naturales podrían delatar de qué fuente viene el compuesto, que es la fuente con menos potencia. Por eso se mira cada procedencia por separado.

In [13]:
por_procedencia = {e: comparacion(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
conc_proc = pd.concat({e: tabla(res, e)["concordancia"].unstack("brazo")[orden_brazos]
                       for e in por_procedencia}, names=["procedencia"])
conc_proc

brazo                        descriptores  depurados  seleccionados  \
procedencia modelo                                                    
solo_chembl arbol                   0.665      0.666          0.665   
            bayesian_ridge          0.681      0.680          0.677   
            knn                     0.733      0.734          0.726   
            linear_svr              0.679      0.678          0.676   
            media                   0.500        NaN            NaN   
            media_por_diana         0.637        NaN            NaN   
            mlp                     0.729      0.731          0.726   
            ridge                   0.680      0.680          0.677   
            svr                     0.730      0.731          0.726   
mixto       arbol                   0.532      0.542          0.544   
            bayesian_ridge          0.510      0.508          0.519   
            knn                     0.565      0.571          0.547   
            linear_svr              0.518      0.517          0.521   
            media                   0.500        NaN            NaN   
            media_por_diana         0.517        NaN            NaN   
            mlp                     0.570      0.576          0.586   
            ridge                   0.515      0.514          0.520   
            svr                     0.564      0.564          0.552   
solo_npass  arbol                   0.498      0.504          0.496   
            bayesian_ridge          0.532      0.533          0.500   
            knn                     0.464      0.465          0.452   
            linear_svr              0.508      0.512          0.493   
            media                   0.500        NaN            NaN   
            media_por_diana         0.403        NaN            NaN   
            mlp                     0.551      0.549          0.520   
            ridge                   0.522      0.524          0.499   
            svr                     0.535      0.532          0.510   

brazo                        descriptores+huellas  huellas  
procedencia modelo                                          
solo_chembl arbol                           0.679    0.711  
            bayesian_ridge                  0.721    0.719  
            knn                             0.749    0.748  
            linear_svr                      0.715    0.713  
            media                             NaN      NaN  
            media_por_diana                   NaN      NaN  
            mlp                             0.706    0.711  
            ridge                           0.718    0.716  
            svr                               NaN      NaN  
mixto       arbol                           0.551    0.574  
            bayesian_ridge                  0.544    0.544  
            knn                             0.554    0.554  
            linear_svr                      0.551    0.556  
            media                             NaN      NaN  
            media_por_diana                   NaN      NaN  
            mlp                             0.554    0.567  
            ridge                           0.548    0.550  
            svr                               NaN      NaN  
solo_npass  arbol                           0.514    0.498  
            bayesian_ridge                  0.490    0.466  
            knn                             0.547    0.548  
            linear_svr                      0.497    0.477  
            media                             NaN      NaN  
            media_por_diana                   NaN      NaN  
            mlp                             0.503    0.511  
            ridge                           0.490    0.466  
            svr                               NaN      NaN

In [14]:
pd.concat(por_procedencia, names=["procedencia"])[["dif_media", "dif_desv", "mejor_en", "mejora"]]

dif_media  dif_desv  mejor_en  mejora
procedencia brazo     modelo                                               
solo_chembl depurados arbol               0.001     0.003         8   False
                      bayesian_ridge     -0.001     0.001         2   False
                      knn                 0.001     0.001        11   False
                      linear_svr         -0.000     0.001         6   False
                      mlp                 0.002     0.003        10   False
...                                         ...       ...       ...     ...
solo_npass  huellas   bayesian_ridge     -0.066     0.023         0   False
                      knn                 0.085     0.022        15    True
                      linear_svr         -0.031     0.020         0   False
                      mlp                -0.040     0.034         3   False
                      ridge              -0.057     0.021         0   False

[78 rows x 4 columns]

### La brecha entre entrenamiento y validación

Un modelo que se aprende los ejemplos acierta mucho en lo que vio y mucho menos en lo que no. La brecha es la concordancia (o el RMSE) sobre una muestra de 5,000 filas de entrenamiento menos la de validación. Con más columnas la brecha crece si el modelo está sobreajustando; que crezca no es un defecto si la validación también mejora. Con el `kNN` la muestra de entrenamiento incluye a cada punto como su propio vecino, así que su brecha sale inflada en todos los brazos y solo se compara dentro del mismo modelo.

In [15]:
t = res[(res["estrato"] == "todos") & ~res["modelo"].isin(list(REFERENCIA))]
brecha = t.groupby(["brazo", "modelo"]).agg(
    conc_entrenamiento=("ent_concordancia", "mean"), conc_validacion=("concordancia", "mean"),
    rmse_entrenamiento=("ent_rmse", "mean"), rmse_validacion=("rmse", "mean"))
brecha["brecha_conc"] = brecha["conc_entrenamiento"] - brecha["conc_validacion"]
brecha["brecha_rmse"] = brecha["rmse_validacion"] - brecha["rmse_entrenamiento"]
brecha[["conc_entrenamiento", "conc_validacion", "brecha_conc", "brecha_rmse"]].unstack("brazo").round(3)

conc_entrenamiento                                            \
brazo                   depurados descriptores descriptores+huellas huellas   
modelo                                                                        
arbol                       0.896        0.897                0.902   0.868   
bayesian_ridge              0.697        0.698                0.776   0.772   
knn                         0.810        0.810                0.817   0.816   
linear_svr                  0.694        0.696                0.777   0.774   
mlp                         0.815        0.823                0.962   0.966   
ridge                       0.696        0.697                0.779   0.776   
svr                         0.764        0.764                  NaN     NaN   

                             conc_validacion               \
brazo          seleccionados       depurados descriptores   
modelo                                                      
arbol                  0.889           0.672        0.670   
bayesian_ridge         0.693           0.692        0.692   
knn                    0.806           0.739        0.739   
linear_svr             0.692           0.690        0.690   
mlp                    0.800           0.737        0.735   
ridge                  0.693           0.691        0.691   
svr                    0.759           0.739        0.738   

                                                          brecha_conc  \
brazo          descriptores+huellas huellas seleccionados   depurados   
modelo                                                                  
arbol                         0.684   0.716         0.670       0.225   
bayesian_ridge                0.728   0.726         0.688       0.005   
knn                           0.756   0.755         0.731       0.071   
linear_svr                    0.722   0.720         0.687       0.005   
mlp                           0.708   0.714         0.732       0.077   
ridge                         0.724   0.722         0.688       0.005   
svr                             NaN     NaN         0.734       0.025   

                                                                        \
brazo          descriptores descriptores+huellas huellas seleccionados   
modelo                                                                   
arbol                 0.227                0.218   0.152         0.219   
bayesian_ridge        0.006                0.048   0.047         0.005   
knn                   0.072                0.061   0.062         0.075   
linear_svr            0.006                0.056   0.054         0.005   
mlp                   0.088                0.255   0.252         0.067   
ridge                 0.006                0.055   0.053         0.005   
svr                   0.026                  NaN     NaN         0.025   

               brecha_rmse                                            \
brazo            depurados descriptores descriptores+huellas huellas   
modelo                                                                 
arbol                0.835        0.841                0.819   0.562   
bayesian_ridge       0.035        0.042                0.154   0.149   
knn                  0.216        0.217                0.217   0.220   
linear_svr           0.032        0.034                0.181   0.174   
mlp                  0.336        0.345                1.099   1.077   
ridge                0.028        0.031                0.186   0.177   
svr                  0.084        0.085                  NaN     NaN   

                              
brazo          seleccionados  
modelo                        
arbol                  0.804  
bayesian_ridge         0.027  
knn                    0.221  
linear_svr             0.028  
mlp                    0.300  
ridge                  0.026  
svr                    0.082

In [16]:
# Registro en experimentos/ (decisión 1.3: qué representación reciben los modelos).
from src.tracking import Corrida

with Corrida("comparativa-regresion-huellas", decision="1.3",
             params={"modelos": list(MODELOS), "referencia": list(REFERENCIA), "brazos": list(BRAZOS),
                     "pliegues": 5, "entrena_con": "solo exactas", "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "corte_mcc": CORTE, "n_bits_sortslice": N_BITS,
                     "radio_morgan": 2, "vocabulario": "se aprende en cada ajuste",
                     "depuracion": {"frecuencia_modal": 0.99, "umbral_spearman": 0.95,
                                    "protegidas": ["TPSA", "NumHDonors"]},
                     "seleccion_l1": {"alpha": ALFA_L1, "protegidas": ["TPSA", "NumHDonors"]},
                     # La regla se cerro antes de ejecutar, pero despues de explorar un pliegue del
                     # 04b: no esta pre-registrada.
                     "regla_mejora": {"umbral_concordancia": UMBRAL_MEJORA, "minimo_evaluaciones": MINIMO_PLIEGUES,
                                      "de": 15, "comparacion_decisiva": "agregado contra descriptores",
                                      "estratos": "exploratorios",
                                      "estado": "fijada despues de explorar un pliegue; no pre-registrada"},
                     "svr_excluido_de": list(ANCHOS),
                     "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": {f"{b}/{m}": v for (b, m), v in tabla_todos.to_dict(orient="index").items()},
                "comparacion_todos": {f"{b}/{m}": v for (b, m), v in
                                      comparacion(res).to_dict(orient="index").items()},
                "brecha": {f"{b}/{m}": v for (b, m), v in brecha.to_dict(orient="index").items()},
                **{e: {f"{b}/{m}": v for (b, m), v in tabla(res, e).to_dict(orient="index").items()}
                   for e in por_procedencia}})

print(c.carpeta.name)

2026-10-28-artemio-1.3-comparativa-regresion-huellas


In [17]:
# Las 15 evaluaciones por brazo, modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba, calculadas aquí. El CSV es para rehacer la lectura
# 3.1 y las diferencias pareadas sin repetir la corrida.
# Va después del registro a propósito: escribirlo antes dejaría el árbol sucio al registrar.
SALIDA = RAIZ / "notebooks" / "datos" / "04c_brazos.csv"
res.to_csv(SALIDA, index=False)
sin_converger = res.loc[~res["convergio"], "modelo"].value_counts()
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")
print(f"ajustes sin convergencia: {dict(sin_converger) or 'ninguno'}")


notebooks/datos/04c_brazos.csv · 2,100 filas
ajustes sin convergencia: ninguno


## 3.1 Lectura

**Las huellas mejoran a cinco de los seis modelos que las reciben, y la mejora se concentra en los pares que solo tiene ChEMBL.** La regla con la que se lee es la que se cerro antes de ejecutar: un brazo mejora a un modelo si la diferencia media de concordancia contra `descriptores` supera 0.007 y es positiva en al menos 12 de las 15 evaluaciones. Se fijo despues de explorar un pliegue del 04b, asi que no esta pre-registrada; lo que si se cumple es que no se toco al ver estas tablas.

### El control

El brazo `descriptores` repite la corrida registrada del 04b con diferencia **0.000** de concordancia en los seis modelos que los dos cuadernos comparten, asi que los demas brazos si se pueden comparar con el 04b. `linear_svr` es nuevo aqui y no tiene contraparte.

De paso, el control confirma el umbral: la mayor dispersion de la concordancia entre sus 15 evaluaciones es 0.0072, practicamente el 0.007 que se tomo del 04b.

### Las huellas mejoran, en el agregado

| Modelo | `descriptores` | `descriptores+huellas` | diferencia | `huellas` | diferencia |
|:--|--:|--:|:--|--:|:--|
| `knn` | 0.739 | **0.756** | +0.017, 15 de 15 | 0.755 | +0.016, 15 de 15 |
| `bayesian_ridge` | 0.692 | 0.728 | +0.036, 15 de 15 | 0.726 | +0.033, 15 de 15 |
| `ridge` | 0.691 | 0.724 | +0.033, 15 de 15 | 0.722 | +0.031, 15 de 15 |
| `linear_svr` | 0.690 | 0.722 | +0.032, 15 de 15 | 0.720 | +0.030, 15 de 15 |
| `arbol` | 0.670 | 0.684 | +0.014, 15 de 15 | 0.716 | +0.046, 15 de 15 |
| `mlp` | 0.735 | 0.708 | -0.027, 0 de 15 | 0.714 | -0.021, 0 de 15 |

Los cinco que mejoran lo hacen en las 15 evaluaciones, no en 12 o 13: la regla no queda al limite en ningun caso, y las diferencias son de dos a cinco veces el umbral. El `mlp` es la excepcion y empeora, tambien en las 15.

El `svr` con nucleo no entra a los brazos anchos por costo, asi que **no se sabe** si con huellas superaria al `knn`. La mejor cifra del cuaderno, 0.756, es la mejor de lo que se midio y no de todo lo posible.

### Los fragmentos reemplazan a los descriptores, no se suman

El brazo `huellas` queda a 0.002 o menos del brazo ancho en `knn`, `ridge`, `bayesian_ridge` y `linear_svr`, y lo supera en el `arbol` (0.716 contra 0.684) y en el `mlp` (0.714 contra 0.708). Con los fragmentos dentro, los 201 descriptores no aportan nada medible. Es lo contrario de lo que midio el [03c](03c_Permeabilidad_huellas.ipynb) en la fase 1, donde `descriptores+huellas` supera a `huellas` en los seis modelos, y las dos cosas son compatibles: la permeabilidad depende de propiedades de la molecula entera y la potencia contra una diana depende de que fragmentos tiene.

### Cuanto queda una vez que se sabe la diana

`media_por_diana` predice la potencia media de la diana de cada par y no mira la quimica: da **0.642** de concordancia, contra 0.500 del promedio de todo. De los 0.239 que el `knn` con descriptores separa por encima de 0.5, unos 0.142 los separa ya quien solo sabe contra que diana se midio. Lo que queda sobre ese piso es la parte que aporta la estructura:

| | `descriptores` | `descriptores+huellas` | `huellas` |
|:--|--:|--:|--:|
| `knn` | 0.097 | **0.114** | 0.113 |
| `bayesian_ridge` | 0.050 | 0.086 | 0.084 |
| `ridge` | 0.049 | 0.082 | 0.080 |
| `linear_svr` | 0.048 | 0.080 | 0.078 |
| `arbol` | 0.028 | 0.042 | 0.074 |
| `mlp` | 0.093 | 0.066 | 0.072 |

Las huellas suben la parte quimica de los modelos lineales de unos 0.05 a unos 0.08, y la del `knn` de 0.097 a 0.114. Son ganancias reales, en 15 de 15, pero sobre una base pequena: la mayor parte de la concordancia de cualquier modelo de este cuaderno sigue siendo saber la diana, no la molecula.

### Por procedencia (exploratorio, no decide)

| Procedencia | Que pasa con las huellas | piso | mejor con huellas |
|:--|:--|--:|--:|
| `solo_chembl` | mejoran a cinco de seis en las 15 evaluaciones, como el agregado; el `mlp` empeora | 0.637 | `knn` 0.749 |
| `mixto` | mejoran a cuatro (`arbol`, `ridge`, `bayesian_ridge`, `linear_svr`); el `knn` baja -0.011, mejor en 6 de 15 | 0.517 | `arbol` 0.574 |
| `solo_npass` | **solo el `knn` mejora**: +0.085, 15 de 15; los lineales empeoran en las 15 (`ridge` -0.057, `bayesian_ridge` -0.066) | **0.403** | `knn` 0.548 |

La fila de NPASS hay que leerla con dos cuidados. Primero, **ahi todos los modelos estan cerca del azar**, con o sin huellas: el mejor llega a 0.548, y con descriptores el `knn` daba 0.464, por debajo de 0.5. La subida del `knn` es la mayor del cuaderno en diferencia y lo deja apenas por encima del azar. Segundo, ahi el piso por diana esta **por debajo** del azar (0.403): la potencia media que una diana tiene en el entrenamiento, dominado por ChEMBL, ordena al reves los pares de NPASS. Por eso las diferencias contra el piso salen grandes en NPASS sin que eso signifique que ahi se aprende mas quimica.

Dicho de otro modo: la ganancia amplia, de cinco de seis modelos, esta en los pares de ChEMBL; en los de productos naturales es de un solo modelo y no los saca de cerca del azar. Es el estrato que al proyecto mas le importa, y es el que las huellas no resuelven. El 03c encontro el mismo sesgo en la fase 1, por el lado de los compuestos sin equivalente natural.

### Quitar descriptores no cambia nada

`depurados` deja 152-157 de los 201 y `seleccionados` 91-102. Ninguno de los dos mejora a ningun modelo: las diferencias de `depurados` van de -0.000 a +0.003, y `seleccionados` empeora un poco a todos, con el `knn` en -0.007, justo en el limite del umbral y peor en las 15 evaluaciones. De los 201 descriptores se puede prescindir de una cuarta parte sin perder nada medible, y no se gana nada por hacerlo. El Lasso conserva 57 en las 15 evaluaciones, entre ellos TPSA y los donadores de hidrogeno, que iban protegidos.

### La brecha entre entrenamiento y validacion

Las huellas ensanchan la brecha de los modelos lineales de 0.006 a unos 0.05 de concordancia, y aun asi su validacion mejora: es capacidad que se usa, no solo memoria. El `mlp` es el caso contrario, y explica por que es el unico que empeora: su brecha pasa de 0.088 a **0.255** (0.962 en entrenamiento contra 0.708 en validacion). El `arbol` ya era el mas sobreajustado con descriptores (0.227) y con `huellas` la baja a 0.152. La del `knn` sale inflada en todos los brazos porque cada punto de la muestra de entrenamiento es su propio vecino, asi que solo se compara contra si mismo.

### Tiempo

La corrida tomo 120 minutos de reloj y 1.43 h de ajustes, sin ningun ajuste sin converger. Los modelos lineales y el `knn` se quedan por debajo de 8 s por ajuste incluso con 2,249 columnas. El costo esta en el `mlp` (18 s con descriptores, 70 s con `descriptores+huellas`) y en el `svr`, que con descriptores ya tarda 56 s y por eso quedo fuera de los brazos anchos. La columna cronometra solo el ajuste y con un hilo: el `knn` sale casi en cero porque su costo esta en predecir.

### Para el Avance 5

- **El 05b entra con huellas.** El mejor individual medido es el `knn` con `descriptores+huellas` (0.756); el brazo `huellas` rinde casi igual con menos columnas, y elegir entre los dos es del 05b.
- **El `mlp` no entra con huellas sin regularizarlo**, y si entra a un stacking hay que mirar su brecha.
- **El `svr` con huellas no esta medido.** Si el 05b lo quiere como base, tiene que pagar su costo o usar una aproximacion lineal, y decirlo.
- **Lo que las huellas no arreglan es el estrato que importa.** Cualquier ensamble de la fase 2 se compara contra 0.642, no contra 0.5, y en los pares que solo tiene NPASS se parte de cerca del azar.


# Declaración de uso de IA y contribución

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Nos apoyó con la generación de código de las huellas y de la depuración dentro del pliegue, y con el diseño de la comparación | Las secciones que nos ayudó a implementar fueron celdas de código | La decisión de medir las huellas, la regla de mejora, la revisión de los resultados y los comentarios generales fueron del equipo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | *Por completar* |
| Artemio Santiago Padilla Robles | *Por completar* |
| José Antonio Vázquez Martínez | *Por completar* |